# Building Retrofit Dataset - Exploratory Data Analysis

This notebook provides an overview and analysis of the integrated building retrofit dataset for PhD research on AI- and IoT-driven optimization.

## Dataset Overview

The dataset integrates:
- **IoT Sensor Data**: Real-time energy, IEQ, weather, and occupancy measurements
- **Building Attributes**: Geometric, structural, and thermal properties
- **Energy Performance**: Historical consumption and efficiency ratings
- **Lifecycle Assessment**: Environmental impacts and carbon footprint data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set visualization styles
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load and Inspect the Data

In [ ]:
# Load the integrated dataset
buildings = pd.read_csv('../data/raw/integrated_building_data.csv')

print(f"Dataset shape: {buildings.shape}")
print(f"\nColumn types:")
print(buildings.dtypes.value_counts())

# Display basic information
buildings.info()

In [ ]:
# Display first few rows
buildings.head()

## 2. Building Stock Analysis

In [ ]:
# Building type distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Building types
buildings['building_type'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Building Type Distribution')
axes[0].set_xlabel('Building Type')
axes[0].set_ylabel('Count')

# Construction periods
buildings['construction_period'].value_counts().sort_index().plot(kind='bar', ax=axes[1])
axes[1].set_title('Construction Period Distribution')
axes[1].set_xlabel('Period')
axes[1].set_ylabel('Count')

# Energy ratings
buildings['energy_rating'].value_counts().sort_index().plot(kind='bar', ax=axes[2], color='green')
axes[2].set_title('Energy Rating Distribution')
axes[2].set_xlabel('Rating (A-G)')
axes[2].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Building size analysis
fig = px.box(buildings, x='building_type', y='gross_floor_area_m2', 
             title='Building Size Distribution by Type',
             labels={'gross_floor_area_m2': 'Gross Floor Area (m²)'})
fig.show()

# Summary statistics
print("Building Size Statistics:")
print(buildings.groupby('building_type')['gross_floor_area_m2'].describe().round(2))

## 3. Energy Performance Analysis

In [ ]:
# Load energy performance data
historical_energy = pd.read_csv('../data/raw/energy_performance/historical_consumption.csv')
energy_ratings = pd.read_csv('../data/raw/energy_performance/energy_ratings.csv')

# Calculate annual energy consumption by building
annual_energy = historical_energy.groupby(['building_id', 'year']).agg({
    'total_kwh': 'sum',
    'electricity_kwh': 'sum',
    'gas_kwh': 'sum',
    'carbon_emissions_kg_co2': 'sum'
}).reset_index()

# Merge with building data
annual_energy = annual_energy.merge(
    buildings[['building_id', 'building_type', 'gross_floor_area_m2']], 
    on='building_id'
)

# Calculate EUI
annual_energy['eui_kwh_m2'] = annual_energy['total_kwh'] / annual_energy['gross_floor_area_m2']

print(f"Energy consumption data shape: {annual_energy.shape}")
annual_energy.head()

In [ ]:
# Energy Use Intensity by building type
fig = px.box(annual_energy, x='building_type', y='eui_kwh_m2',
             title='Energy Use Intensity Distribution by Building Type',
             labels={'eui_kwh_m2': 'EUI (kWh/m²/year)'})
fig.show()

# Time series of energy consumption
energy_trend = annual_energy.groupby('year').agg({
    'total_kwh': 'mean',
    'carbon_emissions_kg_co2': 'mean'
}).reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(x=energy_trend['year'], y=energy_trend['total_kwh'],
               name="Energy Consumption", line=dict(color='blue')),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=energy_trend['year'], y=energy_trend['carbon_emissions_kg_co2'],
               name="Carbon Emissions", line=dict(color='red')),
    secondary_y=True,
)
fig.update_xaxes(title_text="Year")
fig.update_yaxes(title_text="Average Energy (kWh)", secondary_y=False)
fig.update_yaxes(title_text="Average CO2 (kg)", secondary_y=True)
fig.update_layout(title="Energy Consumption and Carbon Emissions Trends")
fig.show()

## 4. Retrofit Potential Analysis

In [ ]:
# Analyze retrofit potential
retrofit_analysis = buildings.groupby(['retrofit_potential', 'building_type']).size().unstack(fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Retrofit potential distribution
buildings['retrofit_potential'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%')
axes[0].set_title('Retrofit Potential Distribution')
axes[0].set_ylabel('')

# Retrofit potential by building type
retrofit_analysis.T.plot(kind='bar', stacked=True, ax=axes[1])
axes[1].set_title('Retrofit Potential by Building Type')
axes[1].set_xlabel('Building Type')
axes[1].set_ylabel('Count')
axes[1].legend(title='Retrofit Potential')

plt.tight_layout()
plt.show()

In [ ]:
# Load retrofit savings data if available
try:
    retrofit_savings = pd.read_csv('../data/raw/energy_performance/retrofit_savings.csv')
    
    # Analyze retrofit measures and savings
    avg_savings = retrofit_savings.groupby('building_id').agg({
        'savings_percentage': 'mean',
        'energy_savings_kwh': 'sum',
        'carbon_savings_kg_co2': 'sum'
    }).reset_index()
    
    print(f"Average savings from retrofits:")
    print(f"  Energy savings: {avg_savings['savings_percentage'].mean():.1f}%")
    print(f"  Total energy saved: {avg_savings['energy_savings_kwh'].sum():,.0f} kWh")
    print(f"  Total carbon saved: {avg_savings['carbon_savings_kg_co2'].sum():,.0f} kg CO2")
    
    # Visualize savings distribution
    fig = px.histogram(avg_savings, x='savings_percentage', nbins=20,
                      title='Distribution of Energy Savings from Retrofits',
                      labels={'savings_percentage': 'Savings (%)', 'count': 'Number of Buildings'})
    fig.show()
    
except FileNotFoundError:
    print("No retrofit savings data available")

## 5. Thermal Performance Analysis

In [ ]:
# Analyze thermal properties
thermal_cols = [col for col in buildings.columns if 'u_value' in col]

if thermal_cols:
    # Create thermal performance summary
    thermal_summary = buildings[thermal_cols + ['construction_period', 'energy_rating']].copy()
    
    # Plot U-values by construction period
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    components = ['wall', 'roof', 'floor', 'window']
    for idx, component in enumerate(components):
        ax = axes[idx // 2, idx % 2]
        col_name = f'thermal_u_value_{component}'
        if col_name in buildings.columns:
            buildings.boxplot(column=col_name, by='construction_period', ax=ax)
            ax.set_title(f'U-value {component.capitalize()} by Period')
            ax.set_xlabel('Construction Period')
            ax.set_ylabel('U-value (W/m²K)')
            ax.get_figure().suptitle('')
    
    plt.tight_layout()
    plt.show()
    
    # Correlation with energy rating
    avg_u_value = buildings[thermal_cols].mean(axis=1)
    
    fig = px.scatter(x=avg_u_value, y=buildings['energy_rating'],
                    title='Average U-value vs Energy Rating',
                    labels={'x': 'Average U-value (W/m²K)', 'y': 'Energy Rating'})
    fig.show()

## 6. IoT Sensor Data Analysis (Sample)

In [ ]:
# Load sample IoT data
try:
    energy_iot = pd.read_csv('../data/raw/iot_sensors/energy_data.csv')
    energy_iot['timestamp'] = pd.to_datetime(energy_iot['timestamp'])
    
    # Select a sample building
    sample_building = energy_iot['building_id'].iloc[0]
    building_data = energy_iot[energy_iot['building_id'] == sample_building].copy()
    
    # Resample to daily
    building_data.set_index('timestamp', inplace=True)
    daily_data = building_data.resample('D').agg({
        'total_consumption_kw': 'sum',
        'hvac_consumption_kw': 'sum',
        'lighting_consumption_kw': 'sum',
        'equipment_consumption_kw': 'sum'
    })
    
    # Plot time series
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=daily_data.index, y=daily_data['total_consumption_kw'],
                             mode='lines', name='Total', line=dict(width=2)))
    fig.add_trace(go.Scatter(x=daily_data.index, y=daily_data['hvac_consumption_kw'],
                             mode='lines', name='HVAC', line=dict(dash='dash')))
    fig.add_trace(go.Scatter(x=daily_data.index, y=daily_data['lighting_consumption_kw'],
                             mode='lines', name='Lighting', line=dict(dash='dot')))
    
    fig.update_layout(title=f'Daily Energy Consumption Pattern - {sample_building}',
                     xaxis_title='Date',
                     yaxis_title='Energy (kWh)',
                     hovermode='x')
    fig.show()
    
    # Load IEQ data
    ieq_data = pd.read_csv('../data/raw/iot_sensors/ieq_data.csv')
    ieq_data['timestamp'] = pd.to_datetime(ieq_data['timestamp'])
    
    sample_ieq = ieq_data[ieq_data['building_id'] == sample_building].head(168)  # One week
    
    # Plot IEQ parameters
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Temperature', 'Humidity', 'CO2', 'PM2.5')
    )
    
    fig.add_trace(go.Scatter(x=sample_ieq['timestamp'], y=sample_ieq['temperature_c']),
                 row=1, col=1)
    fig.add_trace(go.Scatter(x=sample_ieq['timestamp'], y=sample_ieq['relative_humidity_pct']),
                 row=1, col=2)
    fig.add_trace(go.Scatter(x=sample_ieq['timestamp'], y=sample_ieq['co2_ppm']),
                 row=2, col=1)
    fig.add_trace(go.Scatter(x=sample_ieq['timestamp'], y=sample_ieq['pm25_ugm3']),
                 row=2, col=2)
    
    fig.update_layout(title=f'Indoor Environmental Quality - {sample_building} (1 week)',
                     showlegend=False, height=600)
    fig.show()
    
except FileNotFoundError:
    print("IoT sensor data not available")

## 7. Lifecycle Assessment Analysis

In [ ]:
# Load LCA data
try:
    lifecycle_impacts = pd.read_csv('../data/raw/lca/lifecycle_impacts.csv')
    
    # Merge with building data
    lca_analysis = lifecycle_impacts.merge(
        buildings[['building_id', 'building_type', 'construction_period', 'gross_floor_area_m2']],
        on='building_id'
    )
    
    # Calculate carbon intensity
    lca_analysis['carbon_intensity'] = (lca_analysis['net_lifecycle_carbon_kg_co2'] / 
                                        lca_analysis['gross_floor_area_m2'] / 
                                        lca_analysis['assessment_period_years'])
    
    # Visualize carbon footprint breakdown
    carbon_components = ['total_embodied_carbon_kg_co2', 'total_operational_carbon_kg_co2', 
                        'total_end_of_life_carbon_kg_co2']
    
    avg_carbon = lca_analysis[carbon_components].mean()
    
    fig = px.pie(values=avg_carbon.values, names=['Embodied', 'Operational', 'End of Life'],
                title='Average Lifecycle Carbon Footprint Breakdown')
    fig.show()
    
    # Carbon intensity by building type
    fig = px.box(lca_analysis, x='building_type', y='carbon_intensity',
                title='Carbon Intensity by Building Type',
                labels={'carbon_intensity': 'Carbon Intensity (kg CO2/m²/year)'})
    fig.show()
    
except FileNotFoundError:
    print("LCA data not available")

## 8. Correlation Analysis for AI/ML Features

In [ ]:
# Select numerical features for correlation analysis
numerical_features = buildings.select_dtypes(include=[np.number]).columns.tolist()

# Remove ID columns
numerical_features = [col for col in numerical_features if 'id' not in col.lower()]

# Calculate correlation matrix
if len(numerical_features) > 0:
    corr_matrix = buildings[numerical_features].corr()
    
    # Find features most correlated with energy performance
    if 'avg_eui' in corr_matrix.columns:
        energy_correlations = corr_matrix['avg_eui'].abs().sort_values(ascending=False)
        
        print("Top 10 features correlated with Energy Use Intensity:")
        print(energy_correlations.head(11)[1:])  # Exclude self-correlation
        
        # Visualize top correlations
        top_features = energy_correlations.head(11).index[1:]
        
        plt.figure(figsize=(10, 8))
        sns.heatmap(corr_matrix.loc[top_features, top_features], 
                   annot=True, fmt='.2f', cmap='coolwarm', center=0)
        plt.title('Correlation Matrix of Top Energy-Related Features')
        plt.tight_layout()
        plt.show()

## 9. Summary Statistics and Data Quality

In [ ]:
# Data completeness
missing_data = buildings.isnull().sum()
missing_pct = (missing_data / len(buildings) * 100).round(2)

data_quality = pd.DataFrame({
    'Missing Values': missing_data,
    'Missing %': missing_pct
})

data_quality = data_quality[data_quality['Missing Values'] > 0].sort_values('Missing %', ascending=False)

if len(data_quality) > 0:
    print("Data Completeness Issues:")
    print(data_quality.head(10))
    
    # Visualize
    plt.figure(figsize=(10, 6))
    data_quality.head(15)['Missing %'].plot(kind='barh')
    plt.xlabel('Missing Data (%)')
    plt.title('Top 15 Features with Missing Data')
    plt.tight_layout()
    plt.show()
else:
    print("No missing data found in the integrated dataset!")

In [ ]:
# Generate summary report
print("=" * 60)
print("BUILDING RETROFIT DATASET SUMMARY")
print("=" * 60)

print(f"\n📊 Dataset Statistics:")
print(f"  • Total buildings: {len(buildings)}")
print(f"  • Total features: {len(buildings.columns)}")
print(f"  • Numerical features: {len(buildings.select_dtypes(include=[np.number]).columns)}")
print(f"  • Categorical features: {len(buildings.select_dtypes(include=['object']).columns)}")

print(f"\n🏢 Building Stock:")
print(f"  • Building types: {buildings['building_type'].nunique()}")
print(f"  • Construction periods: {buildings['construction_period'].nunique()}")
print(f"  • Average floor area: {buildings['gross_floor_area_m2'].mean():,.0f} m²")
print(f"  • Total floor area: {buildings['gross_floor_area_m2'].sum():,.0f} m²")

if 'avg_eui' in buildings.columns:
    print(f"\n⚡ Energy Performance:")
    print(f"  • Average EUI: {buildings['avg_eui'].mean():.1f} kWh/m²/year")
    print(f"  • Energy rating distribution:")
    for rating in sorted(buildings['energy_rating'].unique()):
        count = (buildings['energy_rating'] == rating).sum()
        pct = count / len(buildings) * 100
        print(f"    - {rating}: {count} ({pct:.1f}%)")

print(f"\n♻️ Retrofit Potential:")
for potential in ['high', 'medium', 'low', 'minimal']:
    if potential in buildings['retrofit_potential'].values:
        count = (buildings['retrofit_potential'] == potential).sum()
        pct = count / len(buildings) * 100
        print(f"  • {potential.capitalize()}: {count} buildings ({pct:.1f}%)")

print("\n" + "=" * 60)
print("Ready for AI/ML model development and optimization!")
print("=" * 60)

## Next Steps for PhD Research

With this integrated dataset, you can now:

### 1. **Predictive Modeling**
   - Energy consumption forecasting using time-series models
   - Retrofit savings prediction
   - Building performance classification

### 2. **Optimization Algorithms**
   - Multi-objective optimization for retrofit strategies
   - Cost-benefit analysis of different retrofit measures
   - Portfolio-level optimization for multiple buildings

### 3. **Deep Learning Applications**
   - LSTM/GRU for energy consumption patterns
   - CNN for building type classification from attributes
   - Reinforcement learning for optimal control strategies

### 4. **Research Questions to Explore**
   - What are the key drivers of energy consumption in different building types?
   - How can IoT data improve retrofit decision-making?
   - What is the optimal sequence of retrofit measures for maximum ROI?
   - How do lifecycle carbon impacts vary across building types and ages?

### 5. **Recommended Analysis Tools**
   - **scikit-learn**: For traditional ML models
   - **TensorFlow/PyTorch**: For deep learning
   - **Prophet/ARIMA**: For time-series forecasting
   - **Optuna**: For hyperparameter optimization
   - **DEAP/PyGMO**: For multi-objective optimization